In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
train_df.shape

(2000, 8)

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
train_df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [7]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [8]:
import string

# Create translation table to remove punctuation
translator = str.maketrans('', '', string.punctuation)

clean_prompts = (
    train_df["prompt"]
    .str.lower()
    .str.translate(translator)
)

vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


In [9]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

prompt_id1 = clean_prompts.iloc[0]

words = prompt_id1.split()

# Remove stop words
filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

print(filtered_words)
print("Words left:", len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Words left: 13


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_df["combined_text"] = (
    train_df["prompt"] + " " +
    train_df["A"] + " " +
    train_df["B"] + " " +
    train_df["C"] + " " +
    train_df["D"] + " " +
    train_df["E"]
)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(train_df["combined_text"])

print("Shape:", X.shape)

Shape: (2000, 2762)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

row = train_df.iloc[0]

prompt = row["prompt"]
option_a = row["A"]

prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

similarity = cosine_similarity(
    prompt_vec,
    option_a_vec
)[0][0]

print("Cosine Similarity:", similarity)

Cosine Similarity: 0.27202429519891635


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

correct_predictions = 0
total_rows = len(train_df)

for _, row in train_df.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    prompt_vec = vectorizer.transform([prompt])

    similarities = {}

    # Calculate similarity with each option
    for option_label, option_text in options.items():

        option_vec = vectorizer.transform([option_text])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[option_label] = sim

    # Option with highest similarity
    predicted_answer = max(
        similarities,
        key=similarities.get
    )

    # Compare with actual answer
    if predicted_answer == row["answer"]:
        correct_predictions += 1

accuracy_percentage = (
    correct_predictions / total_rows
) * 100

print(f"Accuracy: {accuracy_percentage:.2f}%")

Accuracy: 13.55%


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

map3_scores = []

for _, row in train_df.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    prompt_vec = vectorizer.transform([prompt])

    similarities = {}

    for label, option_text in options.items():

        option_vec = vectorizer.transform([option_text])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[label] = sim

    # Sort options by similarity
    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]

    actual_answer = row["answer"]

    # MAP@3 score for this row
    if actual_answer in top3:
        rank = top3.index(actual_answer) + 1
        score = 1 / rank
    else:
        score = 0

    map3_scores.append(score)

final_map3 = sum(map3_scores) / len(map3_scores)

print("MAP@3:", final_map3)

MAP@3: 0.2961666666666667
